# DATA09 特征重要性排序与 6:4 测试验证

本 notebook 用于分析已经计算完成的 DATA09 特征 CSV。特征来源为 `DATA09_single_event_feature_extract.ipynb` 和 `DATA09_v0-flow_feature_extraction.ipynb`，本程序不重新计算原始信号特征，只负责合并、划分、重要性排序、模型测试和可视化。

核心设计：

- 合并：读取多个 CSV，自动去除重复输入文件，并取多文件共有的数值特征列。
- 划分：先统计每个标签下的唯一 `source_file_name` 源文件组，再以源文件组为单位做 6:4 划分；同一源文件派生出的多个窗口样本必须全部进入训练集或全部进入测试集。
- 建模：使用带 median imputer 的 RandomForest pipeline，避免缺失值处理泄漏到测试集。
- 排序：主排序使用训练集内分组交叉验证 permutation importance，并辅以 RandomForest impurity importance 的多随机种子稳定性；测试集 permutation importance 仅作为独立测试审计，不参与主排序。
- 输出：保存合并表、源文件组统计表、划分表、测试指标、特征质量表、特征重要性表和可视化图。


## 1. 环境初始化

导入依赖、设置绘图风格、定位项目根目录，并为本次分析创建带时间戳的输出目录。


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import f_classif
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    pass

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

workspace = Path.cwd()
if not (workspace / 'outputs').exists() and (workspace.parent / 'outputs').exists():
    workspace = workspace.parent

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = workspace / 'outputs' / f'DATA09_feature_importance_from_csv_{RUN_TIMESTAMP}'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'workspace: {workspace}')
print(f'RUN_TIMESTAMP: {RUN_TIMESTAMP}')
print(f'OUTPUT_ROOT: {OUTPUT_ROOT}')


## 2. 输入配置

在 `FEATURE_FILES` 中填写已经计算好的特征 CSV。程序会自动跳过重复路径，并固定使用源文件组级 6:4 划分，40% 源文件组作为测试集。


In [ ]:
# =========================
# Input configuration
# =========================

FEATURE_FILES = [
    Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-qj_features_QJ\features_QJ_20260904_114606.csv'),
    Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-flow_features_20260904_121425\features_20260904_121425_part_0001.csv'),
    Path(r'E:\codes\ZZ-BK\outputs\DATA09_v0-flow_features_20260904_121425\features_20260904_121425_part_0001.csv'),
]

LABEL_BY_PATH_KEYWORD = {
    'v0-flow': 'flow',
    'flow': 'flow',
    'BK00': 'BK00',
    'v0-bk': 'BK00',
    'bk': 'BK00',
    'QJ': 'QJ',
    'v0-qj': 'QJ',
    'qj': 'QJ',
}

RANDOM_STATE = 42
TEST_SIZE = 0.40
TOP_N = 30
N_ESTIMATORS = 500
RUN_PERMUTATION_IMPORTANCE = True
PERMUTATION_REPEATS = 3
CV_FOLDS = 3
RF_STABILITY_SEEDS = [11, 23, 37]
PERMUTATION_CANDIDATE_LIMIT = 80
TEST_PERMUTATION_TOP_N = 30
TOPK_COMPARE_VALUES = [5, 10, 20, 30, 50]
CORRELATION_THRESHOLD = 0.95
NEAR_ZERO_STD_THRESHOLD = 1e-12

print(f'Train ratio target: {1 - TEST_SIZE:.0%}')
print(f'Test ratio target:  {TEST_SIZE:.0%}')
print('Configured FEATURE_FILES:')
for p in FEATURE_FILES:
    print(' ', p)


## 3. 工具函数

定义标签识别、CSV 加载、共同数值特征筛选和源文件分组划分等辅助函数。


In [ ]:
# =========================
# Helper functions
# =========================

META_COLUMNS = {
    'source_file_name', 'source_file_path', 'source_format',
    'source_group_name', 'source_channel_name', 'source_detail',
    'label', 'sample_type', 'window_mode',
    'window_id', 'window_start_index', 'window_end_index',
    'window_length_samples', 'window_step_samples', 'window_duration_s', 'window_start_offset_s',
    'window_start_datetime', 'window_start_ms', 'window_end_ms', 'window_n_samples',
    'sample_rate_hz', 'original_sample_rate_hz', 'source_n_samples', 'source_duration_s',
    'starttime_raw', 'arrival_time_raw', 'arrival_time', 'starttime', 'channel_index',
}


def infer_label_from_path(path: Path) -> str:
    text = str(path).lower()
    for key, label in LABEL_BY_PATH_KEYWORD.items():
        if key.lower() in text:
            return label
    return path.parent.name


def normalize_label_value(value: object, path: Path) -> str:
    if pd.isna(value) or str(value).strip() == '':
        return infer_label_from_path(path)
    text = str(value).strip()
    lower = text.lower()
    if lower in {'v0-flow', 'flow'}:
        return 'flow'
    if lower in {'bk', 'bk00', 'v0-bk'}:
        return 'BK00'
    if lower in {'qj', 'v0-qj'}:
        return 'QJ'
    return text


def load_one_feature_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df['feature_csv_path'] = str(path)
    df['feature_csv_name'] = path.name
    if 'label' in df.columns:
        raw_label = df['label']
    elif 'sample_type' in df.columns:
        raw_label = df['sample_type']
    else:
        raw_label = pd.Series([infer_label_from_path(path)] * len(df), index=df.index)
    df['label'] = [normalize_label_value(v, path) for v in raw_label]
    if 'source_file_name' not in df.columns:
        df['source_file_name'] = path.stem
    df['sample_id'] = df['source_file_name'].astype(str)
    if 'window_mode' in df.columns:
        df['sample_id'] = df['sample_id'] + '|' + df['window_mode'].astype(str)
    elif 'window_id' in df.columns:
        df['sample_id'] = df['sample_id'] + '|window_' + df['window_id'].astype(str)
    return df


def deduplicate_paths(paths: list[Path]) -> list[Path]:
    seen = set()
    out = []
    for path in paths:
        resolved = str(path.resolve()) if path.exists() else str(path)
        if resolved in seen:
            print(f'[WARN] duplicate input skipped: {path}')
            continue
        seen.add(resolved)
        out.append(path)
    return out


def find_feature_columns(frames: list[pd.DataFrame]) -> list[str]:
    common = set(frames[0].columns)
    for df in frames[1:]:
        common &= set(df.columns)
    candidates = [c for c in frames[0].columns if c in common and c not in META_COLUMNS and not c.startswith('feature_csv_')]
    numeric_cols = []
    for col in candidates:
        ok = True
        for df in frames:
            s = pd.to_numeric(df[col], errors='coerce')
            if s.notna().sum() == 0:
                ok = False
                break
        if ok:
            numeric_cols.append(col)
    return numeric_cols


def build_source_group_table(frame: pd.DataFrame) -> pd.DataFrame:
    """统计源文件组。BK/QJ 的多个到时窗口来自同一源文件，划分时必须作为整体处理。"""
    group_table = (
        frame.assign(_label=frame['label'].astype(str), _source=frame['source_file_name'].astype(str))
        .groupby(['_label', '_source'], dropna=False)
        .agg(
            rows=('label', 'size'),
            feature_csvs=('feature_csv_name', lambda s: ', '.join(sorted(set(map(str, s))))),
        )
        .reset_index()
        .rename(columns={'_label': 'label', '_source': 'source_file_name'})
        .sort_values(['label', 'source_file_name'])
        .reset_index(drop=True)
    )
    return group_table


def split_by_source_group(frame: pd.DataFrame, test_size: float, random_state: int):
    """按 source_file_name 源文件组划分训练/测试集，禁止退化为窗口行级划分。"""
    labels = sorted(frame['label'].astype(str).unique())
    if len(labels) < 2:
        raise ValueError('至少需要 2 个标签才能进行分类。')

    group_table = build_source_group_table(frame)
    mixed_groups = frame.groupby('source_file_name')['label'].nunique()
    if (mixed_groups > 1).any():
        bad = mixed_groups[mixed_groups > 1].index.astype(str).tolist()[:20]
        raise ValueError(
            '发现同一个 source_file_name 同时属于多个标签，不能安全做源文件组划分。'
            f'请先检查或重命名源文件以保证跨标签唯一。示例: {bad}'
        )

    label_source_counts = group_table.groupby('label')['source_file_name'].nunique()
    too_few = label_source_counts[label_source_counts < 2]
    if not too_few.empty:
        raise ValueError(
            '以下标签的源文件组少于 2 个，不能做源文件级 train/test 划分；'
            f'请补充源文件或从本次分析中移除该标签: {too_few.to_dict()}'
        )

    rng = np.random.default_rng(random_state)
    train_sources: set[str] = set()
    test_sources: set[str] = set()
    split_rows = []

    for label in labels:
        sources = group_table.loc[group_table['label'] == label, 'source_file_name'].astype(str).to_numpy()
        shuffled = sources.copy()
        rng.shuffle(shuffled)
        n_test = int(round(len(shuffled) * test_size))
        n_test = min(max(1, n_test), len(shuffled) - 1)
        test_for_label = set(shuffled[:n_test])
        train_for_label = set(shuffled[n_test:])
        train_sources.update(train_for_label)
        test_sources.update(test_for_label)
        split_rows.append({
            'label': label,
            'source_files_total': int(len(shuffled)),
            'source_files_train': int(len(train_for_label)),
            'source_files_test': int(len(test_for_label)),
            'actual_source_train_ratio': len(train_for_label) / len(shuffled),
            'actual_source_test_ratio': len(test_for_label) / len(shuffled),
        })

    train_idx = frame.index[frame['source_file_name'].astype(str).isin(train_sources)].to_numpy()
    test_idx = frame.index[frame['source_file_name'].astype(str).isin(test_sources)].to_numpy()
    source_split_summary = pd.DataFrame(split_rows)
    return train_idx, test_idx, group_table, source_split_summary


def make_rf_pipeline(random_state: int) -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('rf', RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            random_state=random_state,
            class_weight='balanced_subsample',
            n_jobs=-1,
        )),
    ])


def build_cv_splitter(frame: pd.DataFrame, train_mask: pd.Series, n_splits: int, random_state: int):
    train_frame = frame.loc[train_mask].copy()
    y_train_text = train_frame['label'].astype(str)
    groups_train = train_frame['source_file_name'].astype(str)
    min_class_count = int(y_train_text.value_counts().min())
    usable_splits = max(2, min(n_splits, min_class_count))
    group_label_counts = train_frame.groupby('source_file_name')['label'].nunique()
    label_group_counts = train_frame.groupby('label')['source_file_name'].nunique()
    if (group_label_counts <= 1).all() and (label_group_counts >= usable_splits).all():
        splitter = StratifiedGroupKFold(n_splits=usable_splits, shuffle=True, random_state=random_state)
        return splitter, groups_train.to_numpy(), usable_splits, 'StratifiedGroupKFold'
    splitter = StratifiedKFold(n_splits=usable_splits, shuffle=True, random_state=random_state)
    return splitter, None, usable_splits, 'StratifiedKFold(row-level fallback)'


def high_correlation_summary(x_numeric: pd.DataFrame, threshold: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    corr = x_numeric.corr(method='spearman').abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = (
        upper.stack()
        .rename('abs_spearman_corr')
        .reset_index()
        .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b'})
        .query('abs_spearman_corr >= @threshold')
        .sort_values('abs_spearman_corr', ascending=False)
        .reset_index(drop=True)
    )
    per_feature = pd.concat([pairs['feature_a'], pairs['feature_b']], ignore_index=True).value_counts().rename_axis('feature').reset_index(name='high_corr_partner_count') if len(pairs) else pd.DataFrame({'feature': [], 'high_corr_partner_count': []})
    return pairs, per_feature


def compute_univariate_f_scores(x_train: pd.DataFrame, y_train_array: np.ndarray, features: list[str]) -> pd.DataFrame:
    """用训练集单变量 ANOVA F 检验给每个特征打分。

    F 检验只看单个特征与类别的边际区分能力，计算很快，适合做 permutation importance
    之前的候选筛选。它不能替代模型内特征重要性，因为它忽略特征之间的组合关系。
    """
    imputed = SimpleImputer(strategy='median').fit_transform(x_train.loc[:, features])
    scores, p_values = f_classif(imputed, y_train_array)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    p_values = np.nan_to_num(p_values, nan=1.0, posinf=1.0, neginf=1.0)
    return pd.DataFrame({'feature': features, 'univariate_f_score': scores, 'univariate_f_pvalue': p_values})


def choose_permutation_candidates(ranking_frame: pd.DataFrame, limit: int) -> list[str]:
    """从多个快速排序来源取并集，限制昂贵 permutation importance 的特征数。

    permutation importance 的复杂度近似为：fold 数 * 候选特征数 * repeats * 模型预测成本。
    先用 RF impurity 稳定性和单变量 F 检验筛候选，通常能保留主要信号，同时显著减少运行时间。
    """
    if limit <= 0 or limit >= len(ranking_frame):
        return ranking_frame['feature'].tolist()
    half = max(1, limit // 2)
    candidates = []
    candidates.extend(ranking_frame.sort_values('rf_importance_seed_mean', ascending=False)['feature'].head(half).tolist())
    candidates.extend(ranking_frame.sort_values('univariate_f_score', ascending=False)['feature'].head(limit - half).tolist())
    # 去重后如果不足 limit，再按综合快速排名补齐。
    seen = set()
    unique = []
    for feat in candidates:
        if feat not in seen:
            seen.add(feat)
            unique.append(feat)
    if len(unique) < limit:
        ranked = ranking_frame.sort_values(['rank_fast_combined', 'feature'])['feature'].tolist()
        for feat in ranked:
            if feat not in seen:
                seen.add(feat)
                unique.append(feat)
            if len(unique) >= limit:
                break
    return unique[:limit]


def correlation_pruned_features(ranking_frame: pd.DataFrame, corr_pair_frame: pd.DataFrame, score_col: str, threshold: float) -> list[str]:
    """按分数从高到低选特征，并跳过已经与已选特征高度相关的冗余特征。

    这是一种轻量级排序降维方法：不改变特征本身，只减少强共线特征的重复进入，
    便于得到更紧凑、更可解释的 Top-K 特征子集。
    """
    high_corr = {}
    if corr_pair_frame is not None and len(corr_pair_frame):
        for row in corr_pair_frame.itertuples(index=False):
            if row.abs_spearman_corr >= threshold:
                high_corr.setdefault(row.feature_a, set()).add(row.feature_b)
                high_corr.setdefault(row.feature_b, set()).add(row.feature_a)
    selected = []
    selected_set = set()
    ordered = ranking_frame.sort_values(score_col, ascending=False)['feature'].tolist()
    for feat in ordered:
        if any(feat in high_corr.get(chosen, set()) for chosen in selected_set):
            continue
        selected.append(feat)
        selected_set.add(feat)
    return selected


def evaluate_topk_feature_sets(rankings: dict[str, list[str]], topk_values: list[int]) -> pd.DataFrame:
    """在固定 6:4 划分上比较不同排序/降维策略的 Top-K 分类效果。

    这里不重新调参，只用同一类 RandomForest pipeline 训练不同特征子集，输出 holdout
    balanced accuracy、macro F1 和 accuracy，用于判断“更少特征是否已经足够”。
    """
    rows = []
    for method, ranked_features in rankings.items():
        ranked_features = [f for f in ranked_features if f in feature_cols]
        for k in topk_values:
            chosen = ranked_features[:min(k, len(ranked_features))]
            if not chosen:
                continue
            subset_model = make_rf_pipeline(RANDOM_STATE)
            subset_model.fit(X_train.loc[:, chosen], y_train)
            pred = subset_model.predict(X_test.loc[:, chosen])
            rows.append({
                'method': method,
                'top_k': len(chosen),
                'accuracy': float(accuracy_score(y_test, pred)),
                'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
                'macro_f1': float(f1_score(y_test, pred, average='macro')),
            })
    return pd.DataFrame(rows)

print('Helper functions ready.')


## 4. 特征表读取与合并

读取所有输入 CSV，输出每个文件的行数、列数、类别和源文件数，并合并为统一特征表。


In [ ]:
# =========================
# Step 1: Load CSV files, deduplicate inputs, and merge rows
# =========================

unique_files = deduplicate_paths(FEATURE_FILES)
if not unique_files:
    raise ValueError('FEATURE_FILES is empty after deduplication.')

frames = []
file_summary_rows = []
for path in unique_files:
    df = load_one_feature_csv(path)
    frames.append(df)
    file_summary_rows.append({
        'file': str(path),
        'rows': len(df),
        'columns': len(df.columns),
        'labels': ', '.join(sorted(df['label'].astype(str).unique())),
        'source_files': df['source_file_name'].nunique(),
    })

file_summary = pd.DataFrame(file_summary_rows)
display(file_summary)

labels_found = sorted(set().union(*(set(df['label'].astype(str).unique()) for df in frames)))
print(f'Loaded CSV count: {len(frames)}')
print(f'Labels found: {labels_found}')
if len(labels_found) < 3:
    print('[WARN] Fewer than 3 classes were loaded. Add the missing BK00/QJ/flow CSV to FEATURE_FILES for three-class testing.')

feature_cols = find_feature_columns(frames)
print(f'Common numeric feature columns: {len(feature_cols)}')
if not feature_cols:
    raise ValueError('No common numeric feature columns found across input CSV files.')

combined = pd.concat(frames, ignore_index=True, sort=False)
combined_path = OUTPUT_ROOT / f'combined_features_{RUN_TIMESTAMP}.csv'
combined.to_csv(combined_path, index=False, encoding='utf-8-sig')
print(f'Saved combined table: {combined_path}')
print(f'Combined shape: {combined.shape}')
display(combined[['label', 'source_file_name', 'sample_id', 'feature_csv_name']].head(10))


## 5. 数据分布与特征质量检查

统计各类别样本数和源文件数，检查共同特征列的缺失率与方差，并绘制类别分布图。


In [ ]:
# =========================
# Step 2: Class distribution and feature quality checks
# =========================

label_counts = combined['label'].value_counts().rename_axis('label').reset_index(name='rows')
source_counts = combined.groupby('label')['source_file_name'].nunique().rename('source_files').reset_index()
summary_counts = label_counts.merge(source_counts, on='label', how='left')
display(summary_counts)

X_all_numeric = combined.loc[:, feature_cols].apply(pd.to_numeric, errors='coerce')
feature_quality = pd.DataFrame({
    'feature': feature_cols,
    'missing_rate': X_all_numeric.isna().mean().to_numpy(),
    'valid_count': X_all_numeric.notna().sum().to_numpy(),
    'mean': X_all_numeric.mean().to_numpy(),
    'std': X_all_numeric.std().to_numpy(),
    'variance': X_all_numeric.var().to_numpy(),
    'unique_count': X_all_numeric.nunique(dropna=True).to_numpy(),
})
feature_quality['near_zero_variance'] = feature_quality['std'].fillna(0) <= NEAR_ZERO_STD_THRESHOLD

corr_pairs, corr_per_feature = high_correlation_summary(X_all_numeric, CORRELATION_THRESHOLD)
feature_quality = feature_quality.merge(corr_per_feature, on='feature', how='left')
feature_quality['high_corr_partner_count'] = feature_quality['high_corr_partner_count'].fillna(0).astype(int)
feature_quality = feature_quality.sort_values(
    ['missing_rate', 'near_zero_variance', 'high_corr_partner_count', 'variance'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

feature_quality_path = OUTPUT_ROOT / f'feature_quality_{RUN_TIMESTAMP}.csv'
feature_quality.to_csv(feature_quality_path, index=False, encoding='utf-8-sig')
print(f'Saved feature quality table: {feature_quality_path}')
print(f'Near-zero variance features: {int(feature_quality["near_zero_variance"].sum())}')
print(f'Highly correlated feature pairs, |Spearman r| >= {CORRELATION_THRESHOLD}: {len(corr_pairs)}')
display(feature_quality.head(20))

if len(corr_pairs):
    corr_pairs_path = OUTPUT_ROOT / f'high_correlation_pairs_{RUN_TIMESTAMP}.csv'
    corr_pairs.to_csv(corr_pairs_path, index=False, encoding='utf-8-sig')
    print(f'Saved high correlation pairs: {corr_pairs_path}')
    display(corr_pairs.head(20))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(summary_counts['label'], summary_counts['rows'], color='#4c78a8')
axes[0].set_title('Rows per class')
axes[0].set_xlabel('label')
axes[0].set_ylabel('rows')
axes[1].bar(summary_counts['label'], summary_counts['source_files'], color='#f58518')
axes[1].set_title('Source files per class')
axes[1].set_xlabel('label')
axes[1].set_ylabel('source files')
plt.tight_layout()
class_plot_path = OUTPUT_ROOT / f'class_distribution_{RUN_TIMESTAMP}.png'
plt.savefig(class_plot_path)
plt.show()
print(f'Saved plot: {class_plot_path}')


## 6. 6:4 训练测试划分

先统计每个标签下有多少个唯一 `source_file_name` 源文件组，再按源文件组做 60% 训练、40% 测试划分。BK、QJ 的一个源文件会派生多个到时窗口样本，这些派生样本具有相似性，因此同一 `source_file_name` 的全部窗口必须进入同一个集合，禁止按截取后的窗口行级划分。


In [ ]:
# =========================
# Step 3: Train/test split grouped by source file
# =========================

train_idx, test_idx, source_group_table, source_split_summary = split_by_source_group(combined, TEST_SIZE, RANDOM_STATE)
combined['split'] = 'train'
combined.loc[test_idx, 'split'] = 'test'

source_group_table_path = OUTPUT_ROOT / f'source_group_table_{RUN_TIMESTAMP}.csv'
source_group_table.to_csv(source_group_table_path, index=False, encoding='utf-8-sig')
source_split_summary_path = OUTPUT_ROOT / f'source_split_summary_{RUN_TIMESTAMP}.csv'
source_split_summary.to_csv(source_split_summary_path, index=False, encoding='utf-8-sig')

print('源文件组统计（按标签）：')
display(source_group_table.groupby('label').agg(
    source_files=('source_file_name', 'nunique'),
    derived_window_rows=('rows', 'sum'),
    rows_per_source_min=('rows', 'min'),
    rows_per_source_max=('rows', 'max'),
).reset_index())

print('源文件组划分结果（划分比例按源文件组计算）：')
display(source_split_summary)
print(f'已保存源文件组统计表: {source_group_table_path}')
print(f'已保存源文件组划分汇总: {source_split_summary_path}')

split_summary = (
    combined.groupby(['split', 'label'], dropna=False)
    .agg(rows=('label', 'size'), source_files=('source_file_name', 'nunique'))
    .reset_index()
    .sort_values(['split', 'label'])
)
print('划分后特征行统计（行数比例仅用于参考，不作为划分依据）：')
display(split_summary)

train_sources = set(combined.loc[combined['split'] == 'train', 'source_file_name'].astype(str))
test_sources = set(combined.loc[combined['split'] == 'test', 'source_file_name'].astype(str))
overlap_sources = sorted(train_sources & test_sources)
if overlap_sources:
    raise RuntimeError(f'源文件组泄漏：{len(overlap_sources)} 个 source_file_name 同时出现在训练集和测试集，示例: {overlap_sources[:20]}')

actual_train_row_ratio = float((combined['split'] == 'train').mean())
actual_test_row_ratio = float((combined['split'] == 'test').mean())
print(f'Train rows: {(combined["split"] == "train").sum()} ({actual_train_row_ratio:.1%}, source-group split target 60%)')
print(f'Test rows:  {(combined["split"] == "test").sum()} ({actual_test_row_ratio:.1%}, source-group split target 40%)')
print(f'Train source files: {len(train_sources)}')
print(f'Test source files:  {len(test_sources)}')
print(f'Overlapped source files between train/test: {len(overlap_sources)}')

split_path = OUTPUT_ROOT / f'combined_features_with_split_{RUN_TIMESTAMP}.csv'
combined.to_csv(split_path, index=False, encoding='utf-8-sig')
print(f'已保存带划分标记的特征表: {split_path}')


## 7. RandomForest 训练与测试

使用训练集拟合 RandomForest 基线模型，并在 40% 测试集上输出指标、分类报告和混淆矩阵。


In [ ]:
# =========================
# Step 4: Train RandomForest and evaluate holdout test set
# =========================

X = combined.loc[:, feature_cols].apply(pd.to_numeric, errors='coerce')
y_text = combined['label'].astype(str)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

train_mask = combined['split'] == 'train'
test_mask = combined['split'] == 'test'

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y[train_mask.to_numpy()]
y_test = y[test_mask.to_numpy()]

model = make_rf_pipeline(RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = {
    'labels': label_encoder.classes_.tolist(),
    'feature_count': len(feature_cols),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred)),
    'macro_f1': float(f1_score(y_test, y_pred, average='macro')),
    'weighted_f1': float(f1_score(y_test, y_pred, average='weighted')),
}
metrics_path = OUTPUT_ROOT / f'test_metrics_{RUN_TIMESTAMP}.json'
metrics_path.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print(f'Saved metrics: {metrics_path}')

report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_path = OUTPUT_ROOT / f'classification_report_{RUN_TIMESTAMP}.csv'
report_df.to_csv(report_path, encoding='utf-8-sig')
display(report_df)
print(f'Saved classification report: {report_path}')

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(label_encoder.classes_)))
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
cm_path = OUTPUT_ROOT / f'confusion_matrix_{RUN_TIMESTAMP}.csv'
cm_df.to_csv(cm_path, encoding='utf-8-sig')
display(cm_df)
print(f'Saved confusion matrix: {cm_path}')

fig, ax = plt.subplots(figsize=(5.5, 4.8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(np.arange(len(label_encoder.classes_)), labels=label_encoder.classes_)
ax.set_yticks(np.arange(len(label_encoder.classes_)), labels=label_encoder.classes_)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix, balanced_acc={metrics["balanced_accuracy"]:.3f}')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
cm_plot_path = OUTPUT_ROOT / f'confusion_matrix_{RUN_TIMESTAMP}.png'
plt.savefig(cm_plot_path)
plt.show()
print(f'Saved plot: {cm_plot_path}')


## 8. 特征重要性排序

重要性排序的科学性风险与修正：

- RandomForest impurity importance 对连续变量、取值更多的变量以及相关特征存在系统性偏好，不能单独作为特征贡献结论。
- 使用测试集 permutation importance 再参与排序，会把独立测试集用于模型解释和筛选，削弱 6:4 holdout 的独立验证意义。
- 单次随机划分和单个随机种子的排序缺少稳定性估计，Top 特征可能只是抽样或模型随机性的结果。
- 高相关特征会分摊或替代彼此的重要性，单个特征排名不等同于物理机制或因果贡献。
- 当类别、来源文件或窗口数量不均衡时，必须优先看 balanced accuracy、macro F1、分组划分和分组交叉验证结果。

效率优化：permutation importance 是本单元最慢部分，复杂度近似为 `fold 数 × 特征数 × repeats × 模型预测成本`。因此本单元先用全量但快速的 RF 多随机种子稳定性和单变量 F 检验做候选筛选，再只对默认前 80 个候选做训练集内 CV permutation importance。未进入候选集的特征不会直接做置换，表中会保留其 RF/F 检验排序，主图也会在 CV permutation 全为 0 时自动切换到 RF 稳定性图，避免“空图”误判。

新增降维对比：增加“相关性去冗余 Top-K”排序。它按快速重要性从高到低选特征，同时跳过与已选特征 Spearman 相关系数过高的特征，用更少、更不冗余的特征子集与现有 RF/F/CV permutation 排序做 holdout 效果对比。


In [ ]:
# =========================
# Step 5: Feature importance ranking with efficient train-only evidence
# =========================

# 1) 读取第 7 节已经训练好的 holdout 模型重要性。
#    这个值来自最终训练模型，只作为参考，不单独作为科学结论。
rf = model.named_steps['rf']
rf_importance = pd.DataFrame({
    'feature': feature_cols,
    'rf_importance_holdout_model': rf.feature_importances_,
})

# 2) 多随机种子 RF 稳定性：仍然覆盖全量特征，但只需训练少量模型，
#    比对 640 个特征逐一 permutation 快得多。
print('Estimating RF impurity-importance stability across random seeds on the training set...')
seed_importances = []
for seed in RF_STABILITY_SEEDS:
    seed_model = make_rf_pipeline(seed)
    seed_model.fit(X_train, y_train)
    seed_importances.append(seed_model.named_steps['rf'].feature_importances_)
seed_importances = np.vstack(seed_importances)
rf_stability = pd.DataFrame({
    'feature': feature_cols,
    'rf_importance_seed_mean': seed_importances.mean(axis=0),
    'rf_importance_seed_std': seed_importances.std(axis=0, ddof=1) if len(RF_STABILITY_SEEDS) > 1 else np.zeros(len(feature_cols)),
})

# 3) 单变量 F 检验：计算快，适合作为候选筛选的第二视角。
#    它只反映单个特征的边际区分能力，不代表组合建模贡献。
univariate_scores = compute_univariate_f_scores(X_train, y_train, feature_cols)

fast_ranking = (
    rf_importance
    .merge(rf_stability, on='feature', how='left')
    .merge(univariate_scores, on='feature', how='left')
    .merge(feature_quality[['feature', 'missing_rate', 'near_zero_variance', 'high_corr_partner_count']], on='feature', how='left')
)
fast_ranking['rank_rf_stability'] = fast_ranking['rf_importance_seed_mean'].rank(ascending=False, method='min').astype(int)
fast_ranking['rank_univariate_f'] = fast_ranking['univariate_f_score'].rank(ascending=False, method='min').astype(int)
fast_ranking['rank_fast_combined'] = (fast_ranking['rank_rf_stability'] + fast_ranking['rank_univariate_f']) / 2.0

# 4) 只对候选特征做训练集内 CV permutation importance。
#    这样保留 permutation 的严谨性，同时把最耗时部分从 640 个特征降到默认 80 个候选。
permutation_candidates = choose_permutation_candidates(fast_ranking, PERMUTATION_CANDIDATE_LIMIT)
print(
    f'Running train-only CV permutation importance on {len(permutation_candidates)} '
    f'candidate features out of {len(feature_cols)} total features...'
)
cv_splitter, cv_groups, cv_splits, cv_name = build_cv_splitter(combined, train_mask, CV_FOLDS, RANDOM_STATE)
print(f'CV splitter: {cv_name}, folds={cv_splits}, repeats={PERMUTATION_REPEATS}')
cv_perm_rows = []
if cv_groups is None:
    split_iter = cv_splitter.split(X_train, y_train)
else:
    split_iter = cv_splitter.split(X_train, y_train, groups=cv_groups)

for fold, (fold_train_pos, fold_valid_pos) in enumerate(split_iter, start=1):
    fold_model = make_rf_pipeline(RANDOM_STATE + fold)
    fold_model.fit(X_train.iloc[fold_train_pos].loc[:, permutation_candidates], y_train[fold_train_pos])
    fold_pred = fold_model.predict(X_train.iloc[fold_valid_pos].loc[:, permutation_candidates])
    fold_score = balanced_accuracy_score(y_train[fold_valid_pos], fold_pred)
    perm = permutation_importance(
        fold_model,
        X_train.iloc[fold_valid_pos].loc[:, permutation_candidates],
        y_train[fold_valid_pos],
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE + fold,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )
    cv_perm_rows.append(pd.DataFrame({
        'fold': fold,
        'feature': permutation_candidates,
        'cv_permutation_importance': perm.importances_mean,
        'cv_permutation_importance_std_within_fold': perm.importances_std,
        'fold_balanced_accuracy': fold_score,
        'fold_valid_rows': len(fold_valid_pos),
    }))

cv_perm_long = pd.concat(cv_perm_rows, ignore_index=True)
cv_perm_long_path = OUTPUT_ROOT / f'cv_permutation_importance_long_{RUN_TIMESTAMP}.csv'
cv_perm_long.to_csv(cv_perm_long_path, index=False, encoding='utf-8-sig')
print(f'Saved CV permutation long table: {cv_perm_long_path}')

cv_perm = (
    cv_perm_long.groupby('feature')
    .agg(
        cv_permutation_importance_mean=('cv_permutation_importance', 'mean'),
        cv_permutation_importance_std=('cv_permutation_importance', 'std'),
        cv_permutation_positive_fold_fraction=('cv_permutation_importance', lambda s: float((s > 0).mean())),
        cv_fold_balanced_accuracy_mean=('fold_balanced_accuracy', 'mean'),
    )
    .reset_index()
)
cv_perm['cv_permutation_importance_std'] = cv_perm['cv_permutation_importance_std'].fillna(0)

# 5) 测试集 permutation 只做审计，而且只对主排序前若干特征计算，
#    避免再次扫描全量 640 特征。
test_perm_df = pd.DataFrame({'feature': feature_cols})
if RUN_PERMUTATION_IMPORTANCE and len(permutation_candidates):
    audit_features = permutation_candidates[:min(TEST_PERMUTATION_TOP_N, len(permutation_candidates))]
    print(f'Running holdout-test permutation audit on top {len(audit_features)} candidate features only...')
    audit_model = make_rf_pipeline(RANDOM_STATE)
    audit_model.fit(X_train.loc[:, audit_features], y_train)
    test_perm = permutation_importance(
        audit_model,
        X_test.loc[:, audit_features],
        y_test,
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )
    test_perm_df = pd.DataFrame({
        'feature': audit_features,
        'test_permutation_importance_mean_audit_only': test_perm.importances_mean,
        'test_permutation_importance_std_audit_only': test_perm.importances_std,
    })

importance = (
    fast_ranking
    .merge(cv_perm, on='feature', how='left')
    .merge(test_perm_df, on='feature', how='left')
)
importance['in_permutation_candidate_set'] = importance['feature'].isin(permutation_candidates)
importance['cv_permutation_importance_mean'] = importance['cv_permutation_importance_mean'].fillna(0.0)
importance['cv_permutation_importance_std'] = importance['cv_permutation_importance_std'].fillna(0.0)
importance['cv_permutation_positive_fold_fraction'] = importance['cv_permutation_positive_fold_fraction'].fillna(0.0)
importance['stability_adjusted_score'] = importance['cv_permutation_importance_mean'] - importance['cv_permutation_importance_std']
importance['rank_cv_permutation'] = importance['cv_permutation_importance_mean'].rank(ascending=False, method='min').astype(int)
importance['rank_stability_adjusted'] = importance['stability_adjusted_score'].rank(ascending=False, method='min').astype(int)

# 如果 permutation 全为 0，说明模型在当前数据上存在强冗余或分类太容易；
# 此时主排序退回快速综合排序，并在表中保留 permutation=0 的事实。
all_cv_perm_zero = bool(np.isclose(importance['cv_permutation_importance_mean'].abs().max(), 0.0))
if all_cv_perm_zero:
    print('[WARN] All CV permutation importance values are zero. Ranking falls back to RF/F-score fast ranking.')
    sort_cols = ['rank_fast_combined', 'rank_rf_stability', 'rank_univariate_f', 'feature']
else:
    sort_cols = ['rank_stability_adjusted', 'rank_cv_permutation', 'rank_fast_combined', 'feature']
importance = importance.sort_values(sort_cols).reset_index(drop=True)
importance.insert(0, 'importance_rank', np.arange(1, len(importance) + 1))

importance_path = OUTPUT_ROOT / f'feature_importance_{RUN_TIMESTAMP}.csv'
importance.to_csv(importance_path, index=False, encoding='utf-8-sig')
print(f'Saved feature importance: {importance_path}')
display(importance.head(TOP_N))

# 6) 新增一种排序降维方式：相关性去冗余排序，
#    并与 RF、F 检验、CV permutation 排序做 Top-K 对比。
cv_ranked_features = importance.sort_values(['rank_stability_adjusted', 'rank_fast_combined', 'feature'])['feature'].tolist()
rf_ranked_features = importance.sort_values(['rank_rf_stability', 'feature'])['feature'].tolist()
f_ranked_features = importance.sort_values(['rank_univariate_f', 'feature'])['feature'].tolist()
corr_pruned_ranked_features = correlation_pruned_features(importance, corr_pairs, 'rf_importance_seed_mean', CORRELATION_THRESHOLD)
ranking_compare = evaluate_topk_feature_sets({
    'cv_permutation_then_fast_fallback': cv_ranked_features,
    'rf_stability': rf_ranked_features,
    'univariate_f_score': f_ranked_features,
    'correlation_pruned_rf': corr_pruned_ranked_features,
}, TOPK_COMPARE_VALUES)
ranking_compare_path = OUTPUT_ROOT / f'feature_ranking_topk_comparison_{RUN_TIMESTAMP}.csv'
ranking_compare.to_csv(ranking_compare_path, index=False, encoding='utf-8-sig')
print(f'Saved feature ranking Top-K comparison: {ranking_compare_path}')
display(ranking_compare.sort_values(['balanced_accuracy', 'macro_f1', 'top_k'], ascending=[False, False, True]))

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for method, grp in ranking_compare.groupby('method'):
    grp = grp.sort_values('top_k')
    ax.plot(grp['top_k'], grp['balanced_accuracy'], marker='o', linewidth=1.8, label=method)
ax.set_xlabel('Top-K selected features')
ax.set_ylabel('Holdout balanced accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('Feature ranking / dimensionality reduction comparison')
ax.legend(loc='best')
plt.tight_layout()
compare_plot_path = OUTPUT_ROOT / f'feature_ranking_topk_comparison_{RUN_TIMESTAMP}.png'
plt.savefig(compare_plot_path)
plt.show()
print(f'Saved plot: {compare_plot_path}')

# 7) 主图：优先画 CV permutation；若数值全为 0，则改画 RF 稳定性，
#    避免因为所有条形长度为 0 而出现“看不到条形图”的误判。
fig, ax = plt.subplots(figsize=(9, max(5, TOP_N * 0.24)))
plot_df = importance.head(TOP_N).iloc[::-1]
if all_cv_perm_zero:
    ax.barh(plot_df['feature'], plot_df['rf_importance_seed_mean'], color='#4c78a8', xerr=plot_df['rf_importance_seed_std'])
    ax.set_xlabel('RF impurity importance mean across seeds')
    ax.set_title(f'Top {TOP_N} Feature Importance, RF fallback because CV permutation is all zero')
else:
    ax.barh(plot_df['feature'], plot_df['cv_permutation_importance_mean'], color='#4c78a8', xerr=plot_df['cv_permutation_importance_std'])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Train-CV permutation importance, balanced accuracy drop')
    ax.set_title(f'Top {TOP_N} Feature Importance, train-CV stability adjusted')
plt.tight_layout()
importance_plot_path = OUTPUT_ROOT / f'feature_importance_top{TOP_N}_{RUN_TIMESTAMP}.png'
plt.savefig(importance_plot_path)
plt.show()
print(f'Saved plot: {importance_plot_path}')


## 9. 错误样本与 Top 特征分布

列出测试集中分类错误的样本，并绘制高重要性特征在各类别中的分布。


In [ ]:
# =========================
# Step 6: Misclassified samples and original count-based top feature distributions
# =========================

# 统计测试集错误样本，定位模型在独立 holdout 上最容易混淆的来源文件。
test_rows = combined.loc[test_mask, ['label', 'source_file_name', 'sample_id', 'feature_csv_name']].copy()
test_rows['pred_label'] = label_encoder.inverse_transform(y_pred)
test_rows['correct'] = test_rows['label'].astype(str) == test_rows['pred_label'].astype(str)
misclassified = test_rows.loc[~test_rows['correct']].copy()
misclassified_path = OUTPUT_ROOT / f'misclassified_samples_{RUN_TIMESTAMP}.csv'
misclassified.to_csv(misclassified_path, index=False, encoding='utf-8-sig')
print(f'Misclassified samples: {len(misclassified)}')
print(f'Saved misclassified samples: {misclassified_path}')
display(misclassified.head(50))

# 保留原始计数直方图：纵轴是样本数，适合看绝对数量；
# 但类别样本不均衡时，小类的柱子会被大类压低。
plot_features = importance.head(min(6, len(importance)))['feature'].tolist()
if plot_features:
    n = len(plot_features)
    fig, axes = plt.subplots(n, 1, figsize=(8, max(3, 2.4 * n)), sharex=False)
    if n == 1:
        axes = [axes]
    for ax, feat in zip(axes, plot_features):
        all_values = pd.to_numeric(combined[feat], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if all_values.empty:
            ax.text(0.5, 0.5, 'all values are NaN', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(feat)
            continue

        positive = all_values[all_values > 0]
        use_log = False
        if len(positive) == len(all_values):
            q01 = positive.quantile(0.01)
            q99 = positive.quantile(0.99)
            use_log = bool(q01 > 0 and q99 / q01 > 1e4)

        plotted_any = False
        for label in label_encoder.classes_:
            raw = pd.to_numeric(
                combined.loc[combined['label'].astype(str) == label, feat],
                errors='coerce',
            ).replace([np.inf, -np.inf], np.nan).dropna()
            if raw.empty:
                continue
            values = np.log10(raw.to_numpy()) if use_log else raw.to_numpy()
            ax.hist(values, bins=30, alpha=0.45, density=False, label=f'{label} (n={len(raw)})')
            plotted_any = True

        if not plotted_any:
            ax.text(0.5, 0.5, 'no finite values to plot', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(feat + (' [log10 scale]' if use_log else ''))
        ax.set_ylabel('count')
        ax.legend(loc='best')
    axes[-1].set_xlabel('feature value')
    plt.tight_layout()
    dist_plot_path = OUTPUT_ROOT / f'top_feature_distributions_{RUN_TIMESTAMP}.png'
    plt.savefig(dist_plot_path)
    plt.show()
    print(f'Saved plot: {dist_plot_path}')


## 10. Top 特征分布的均衡可视化补充

上一节保留原始计数直方图，适合观察每类的绝对样本量。但当 flow 与 QJ/BK00 样本数差异很大时，数量少的类别柱子会显得很低，难以比较分布形状。本节新增归一化密度直方图和经验累积分布函数（ECDF）：每个类别各自归一化，不再让样本数差异主导图形高度，更适合比较不同类别在同一特征上的取值范围、峰值位置和分布偏移。


In [ ]:
# =========================
# Step 7: Balanced top-feature distribution plots
# =========================

plot_features_balanced = importance.head(min(6, len(importance)))['feature'].tolist()
if plot_features_balanced:
    n = len(plot_features_balanced)
    fig, axes = plt.subplots(n, 2, figsize=(12, max(3.2, 2.6 * n)), sharex=False)
    if n == 1:
        axes = np.array([axes])

    for row_idx, feat in enumerate(plot_features_balanced):
        ax_hist = axes[row_idx, 0]
        ax_ecdf = axes[row_idx, 1]
        all_values = pd.to_numeric(combined[feat], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if all_values.empty:
            ax_hist.text(0.5, 0.5, 'all values are NaN', ha='center', va='center', transform=ax_hist.transAxes)
            ax_ecdf.text(0.5, 0.5, 'all values are NaN', ha='center', va='center', transform=ax_ecdf.transAxes)
            continue

        positive = all_values[all_values > 0]
        use_log = False
        if len(positive) == len(all_values):
            q01 = positive.quantile(0.01)
            q99 = positive.quantile(0.99)
            use_log = bool(q01 > 0 and q99 / q01 > 1e4)

        # 统一 bins，保证不同类别的密度曲线在相同横轴区间比较。
        all_plot_values = np.log10(all_values.to_numpy()) if use_log else all_values.to_numpy()
        bins = np.histogram_bin_edges(all_plot_values, bins=30)

        for label in label_encoder.classes_:
            raw = pd.to_numeric(
                combined.loc[combined['label'].astype(str) == label, feat],
                errors='coerce',
            ).replace([np.inf, -np.inf], np.nan).dropna()
            if raw.empty:
                continue
            values = np.log10(raw.to_numpy()) if use_log else raw.to_numpy()

            # density=True 让每个类别面积归一化为 1，解决类别数量不均衡导致的小类柱子不可见问题。
            ax_hist.hist(values, bins=bins, alpha=0.35, density=True, label=f'{label} (n={len(raw)})')

            # ECDF 显示每类样本在不同特征值以下的累计比例，适合比较分布整体偏移。
            sorted_values = np.sort(values)
            ecdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
            ax_ecdf.step(sorted_values, ecdf, where='post', linewidth=1.8, label=f'{label} (n={len(raw)})')

        title_suffix = ' [log10 scale]' if use_log else ''
        ax_hist.set_title(f'{feat} density{title_suffix}')
        ax_hist.set_ylabel('density')
        ax_hist.legend(loc='best')
        ax_ecdf.set_title(f'{feat} ECDF{title_suffix}')
        ax_ecdf.set_ylabel('cumulative fraction')
        ax_ecdf.set_ylim(0, 1.02)
        ax_ecdf.legend(loc='best')

    axes[-1, 0].set_xlabel('feature value')
    axes[-1, 1].set_xlabel('feature value')
    plt.tight_layout()
    balanced_dist_plot_path = OUTPUT_ROOT / f'top_feature_distributions_density_ecdf_{RUN_TIMESTAMP}.png'
    plt.savefig(balanced_dist_plot_path)
    plt.show()
    print(f'Saved plot: {balanced_dist_plot_path}')


## 11. 输出文件清单


In [ ]:
# =========================
# Step 8: Output artifact list
# =========================

artifacts = sorted(OUTPUT_ROOT.glob('*'))
print(f'Artifacts in {OUTPUT_ROOT}:')
for path in artifacts:
    print(' ', path.name)
